# TitanAgent 実験2.5: 小型 Titans (MAC) 長期記憶ストレス＆記憶上書き検証（SFT → GRPO）

実験2では 10.19M の Titans (MAC) が SFT 時点で全レベル 100% に到達し、GRPO がほぼ `skip(zero-var)` になりました。
本ノートブックでは **Titans (MAC) の長期記憶の限界と真価** を測るため、タスクを4段階のストレス設計へ引き上げます（Kaggle 無料枠 T4 16GB OOM 安全・約 1.5〜2 時間完走目安）。

### 4段階の記憶ストレス・カリキュラム
1. **Level 1（短距離・基本記憶再利用）**: distractor 1〜2 件。Turn 1 の前提を聞き直さずに再利用。
2. **Level 2（超長距離の記憶保持）**: distractor 10〜16 件（最大 ~800〜900 tokens、`segment_len=64` を 10 セグメント以上またぐ）を超えて Turn 1 の前提を保持。
3. **Level 3（記憶の上書き・更新）**: Turn 1 で記憶した前提が途中の `更新:` ターンで変更される。古い値ではなく**最新の値**を引き出す（古い値を使うと `stale_memory` ペナルティ）。
4. **Level 4（複合ストレス：2スロット同時記憶＋上書き＋長距離）**: Turn 1 の `制約` と `優先軸` の両方を保持しつつ、途中で片方が上書きされる。Turn 2 ではテーマのみ提示され、2つの記憶を同時に思い出して `目的` だけを1回逆質問して整理。

### 実験2 からの設計改善
- **SFT と GRPO の難易度分離**: SFT は中距離（`d <= 8`）＆ 350 step に留め、GRPO に長距離（`d=10..16`）と複合 L4 の明確な伸びしろを残す。
- **距離別劣化曲線 (`d = 2, 6, 12, 18`)**: distractor 数に応じた記憶保持率の変化を SFT 後・GRPO 後で比較。
- **厳格フォーマット検証**: 実験2で見つかった `| |` 二重パイプもフォーマット違反として検知。

In [ ]:
# 1. 環境確認 & ライブラリインストール
import os, sys, torch

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f'Environment: Colab={IN_COLAB}, Kaggle={IN_KAGGLE}')
print(f'PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

!pip -q install -U titans-pytorch

In [ ]:
# 2. 実験設定（Kaggle T4 16GB / seq_len=1024 で OOM しない安全デフォルト）
from pathlib import Path

PRESET = 'tiny'          # 'micro' (~1.5M), 'tiny' (推奨・~10.2M), 'small' (~22M)
SFT_STEPS = 350          # Stage 1 (SFT): 中距離 (d<=8) 中心で止め、GRPO に伸びしろを残す
GRPO_STEPS = 80          # Stage 2 (GRPO): 長距離 L2 (d=10..16) / 上書き L3 / 複合 L4 を重点強化
GROUP_SIZE = 4           # 1問あたりの生成本数
GRPO_TEMP = 1.0          # グループ内多様性を確保する温度
MAX_NEW_TOKENS = 160     # RL・評価生成の最大長
KL_BETA = 0.04           # SFT基礎を保つKL制約
EVAL_PER_LEVEL = 25      # 4レベル評価の各問題数
EVAL_PER_DIST = 15       # 距離別劣化曲線 (d=2,6,12,18) の各問題数
SEED = 42

if IN_KAGGLE:
    OUTPUT_DIR = Path('/kaggle/working/checkpoints/exp2_5_memory_stress')
else:
    OUTPUT_DIR = Path('./checkpoints/exp2_5_memory_stress')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    'micro': dict(dim=128, depth=3, segment_len=32, num_persist_mem_tokens=4, num_longterm_mem_tokens=8,  seq_len=1024, sft_batch_size=4, sft_grad_accum=2,  sft_lr=1.5e-3, grpo_lr=1e-4),
    'tiny':  dict(dim=256, depth=4, segment_len=64, num_persist_mem_tokens=4, num_longterm_mem_tokens=16, seq_len=1024, sft_batch_size=2, sft_grad_accum=16, sft_lr=8e-4,   grpo_lr=4e-5),
    'small': dict(dim=384, depth=6, segment_len=64, num_persist_mem_tokens=8, num_longterm_mem_tokens=16, seq_len=1024, sft_batch_size=1, sft_grad_accum=32, sft_lr=5e-4,   grpo_lr=2.5e-5),
}
cfg = PRESETS[PRESET]
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Selected preset:', PRESET, cfg)

In [ ]:
# 3. 自前トークナイザー & 4レベル記憶ストレス生成器
import copy, json, math, random, re, time
from contextlib import nullcontext
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from titans_pytorch import MemoryAsContextTransformer

torch.manual_seed(SEED)
random.seed(SEED)

SPECIAL_TOKENS = [
    '<|pad|>', '<|eos|>', '<|unk|>',
    '<think>', '</think>', '<ask>', '</ask>',
    '<user>', '</user>', '<summary>', '</summary>',
]

TOPICS = [
    '副業', 'アプリ開発', '英語学習', '転職',
    'ブログ執筆', '資格勉強', '運動習慣', '読書習慣',
    '個人制作', '研究テーマ', '時間管理', '発信活動',
    '資産形成', 'デザイン学習', '自動化ツール', 'キャリア設計',
]

SLOT_NAMES = ['目的', '制約', '優先軸']

SLOT_VALUES = {
    '目的': ['収益化', 'スキル習得', '習慣化', '実績作り', 'キャリア転換', '好奇心探究', '業務効率化', '自己表現'],
    '制約': ['平日30分', '週末のみ', '予算ゼロ', '1日1時間', '期限1か月', 'スマホ中心', '低負荷', '夜間のみ'],
    '優先軸': ['継続しやすさ', '短期成果', '楽しさ', '質重視', '低リスク', '独学完結'],
}

PRIORITY_TO_PLAN = {
    '継続しやすさ': '最小単位で毎日回す',
    '短期成果': '期限を切り一点集中',
    '楽しさ': '興味軸で小さく試す',
    '質重視': '設計を固めて深掘り',
    '低リスク': '無料枠で検証優先',
    '独学完結': '教材を絞り自走する',
}

CONSULT_TEMPLATES = [
    '相談: {topic}を考えている。{known_text}。どう進めるべき？',
    '相談: {topic}について思考を整理したい。{known_text}。方針を決めたい。',
    '相談: {topic}を始めたいが迷っている。{known_text}。',
]

CONSULT_NO_KNOWN_TEMPLATES = [
    '相談: {topic}を考えている。前の前提を踏まえて方針を決めたい。',
    '相談: {topic}について思考を整理したい。どう進めるべき？',
    '相談: {topic}を始めたいが迷っている。整理してほしい。',
]

UPDATE_TEMPLATES = [
    '更新: 状況が変わり、{slot}は{new_val}に変更した。',
    '更新: 方針を見直し、今後の{slot}は{new_val}にする。',
    '更新: 今の{slot}は{new_val}へ切り替えている。',
]

DISTRACTOR_TURNS = [
    'メモ: 今の考えをそのまま保持中。',
    'メモ: 少し休憩してから次の相談へ移る。',
    'メモ: 別のテーマも続けて壁打ちしたい。',
    'メモ: 頭の中を少しずつ言語化している。',
    'メモ: 今日はタスクの優先順位を見直した。',
    'メモ: 気になっている論点をメモに残しておく。',
    'メモ: 進め方のアイデアをいくつか比較している。',
    'メモ: 細かい条件は後でまとめて確認しよう。',
    'メモ: 作業環境を整えて集中しやすい状態にした。',
    'メモ: 過去のメモを読み返して方向性を確認中。',
]

class ThinkingTokenizer:
    def __init__(self, token_to_id):
        self.token_to_id = dict(token_to_id)
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.pad_token_id = self.token_to_id['<|pad|>']
        self.eos_token_id = self.token_to_id['<|eos|>']
        self.unk_token_id = self.token_to_id['<|unk|>']
        self.ask_end_id = self.token_to_id['</ask>']
        self.vocab_size = len(self.token_to_id)
        self._split_re = re.compile('(' + '|'.join(re.escape(t) for t in SPECIAL_TOKENS) + ')')

    @classmethod
    def build_default(cls):
        chars = set('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz+-*/=|():.,!? \n、。')
        for s in [
            '相談: テーマ: 既知: 記憶: 更新記憶: 不足: 確認: 整理完了。全前提が揃った。',
            '方針=エラー: 該当なし', 'テーマ=目的=制約=優先軸=方針=',
            'を聞く。次はが必要。前回の前提を再利用。最新値を使用。',
            *TOPICS, *SLOT_NAMES,
            *[v for vals in SLOT_VALUES.values() for v in vals],
            *PRIORITY_TO_PLAN.values(), *CONSULT_TEMPLATES,
            *CONSULT_NO_KNOWN_TEMPLATES, *UPDATE_TEMPLATES, *DISTRACTOR_TURNS,
        ]:
            chars.update(s)
        vocab = {}
        for tok in SPECIAL_TOKENS:
            vocab[tok] = len(vocab)
        for ch in sorted(chars):
            if ch not in vocab:
                vocab[ch] = len(vocab)
        return cls(vocab)

    def encode(self, text):
        ids = []
        for part in self._split_re.split(text):
            if not part:
                continue
            if part in self.token_to_id:
                ids.append(self.token_to_id[part])
            else:
                for ch in part:
                    ids.append(self.token_to_id.get(ch, self.unk_token_id))
        return ids

    def decode(self, ids, skip_special_pad=True):
        out = []
        for i in ids:
            tok = self.id_to_token.get(i, '')
            if skip_special_pad and tok == '<|pad|>':
                continue
            out.append(tok)
        return ''.join(out)

@dataclass
class Episode:
    level: int
    prompt: str
    topic: str
    full_slots: dict
    known_slots: list
    memory_slots: list
    updated_slots: dict
    missing_slots: list
    expected_plan: str
    ideal_completion: str
    num_distractors: int

def _fmt_known(slots, keys):
    return '、'.join(f'{k}は{slots[k]}' for k in keys)

def _fmt_kv(slots, keys):
    return '、'.join(f'{k}={slots[k]}' for k in keys)

def _build_summary(topic, slots, plan):
    return f"テーマ={topic} | 目的={slots['目的']} | 制約={slots['制約']} | 優先軸={slots['優先軸']} | 方針={plan}"

def _make_distractors(rng, count):
    return ('\n'.join(rng.choice(DISTRACTOR_TURNS) for _ in range(count)) + '\n') if count > 0 else ''

def _build_turn1(rng, topic1, slots1, asked_slot):
    plan1 = PRIORITY_TO_PLAN[slots1['優先軸']]
    t1_known = [k for k in SLOT_NAMES if k != asked_slot]
    return (
        rng.choice(CONSULT_TEMPLATES).format(topic=topic1, known_text=_fmt_known(slots1, t1_known)) + '\n'
        f'<think>テーマ: {topic1}。既知: {_fmt_kv(slots1, t1_known)}。不足: {asked_slot}</think>'
        f'<ask>{asked_slot}</ask><user>{asked_slot}: {slots1[asked_slot]}</user>'
        f'<think>整理完了。全前提が揃った。</think>'
        f'<summary>{_build_summary(topic1, slots1, plan1)}</summary>\n'
    )

def generate_episode(level, rng, num_distractors=None):
    topic1, topic2 = rng.sample(TOPICS, 2)

    if level in (1, 2):
        n_dist = num_distractors if num_distractors is not None else (rng.randint(1, 2) if level == 1 else rng.randint(6, 12))
        mem_slot = rng.choice(['制約', '優先軸'])
        shared_val = rng.choice(SLOT_VALUES[mem_slot])
        slots1 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        slots1[mem_slot] = shared_val
        turn1 = _build_turn1(rng, topic1, slots1, asked_slot=mem_slot)

        slots2 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        slots2[mem_slot] = shared_val
        remaining = [k for k in SLOT_NAMES if k != mem_slot]
        known2 = rng.choice(remaining)
        missing2 = [k for k in remaining if k != known2][0]
        plan2 = PRIORITY_TO_PLAN[slots2['優先軸']]
        prompt = turn1 + _make_distractors(rng, n_dist) + rng.choice(CONSULT_TEMPLATES).format(topic=topic2, known_text=_fmt_known(slots2, [known2])) + '\n'
        ideal = (
            f'<think>テーマ: {topic2}。既知: {known2}={slots2[known2]}。記憶: {mem_slot}={shared_val}。不足: {missing2}</think>'
            f'<ask>{missing2}</ask><user>{missing2}: {slots2[missing2]}</user>'
            f'<think>整理完了。全前提が揃った。</think>'
            f'<summary>{_build_summary(topic2, slots2, plan2)}</summary><|eos|>'
        )
        return Episode(level, prompt, topic2, slots2, [known2], [mem_slot], {}, [missing2], plan2, ideal, n_dist)

    if level == 3:
        n_dist = num_distractors if num_distractors is not None else rng.randint(4, 8)
        d_before = max(1, n_dist // 2)
        d_after = max(1, n_dist - d_before)
        mem_slot = rng.choice(['制約', '優先軸'])
        old_val, new_val = rng.sample(SLOT_VALUES[mem_slot], 2)
        slots1 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        slots1[mem_slot] = old_val
        turn1 = _build_turn1(rng, topic1, slots1, asked_slot=mem_slot)
        update_line = rng.choice(UPDATE_TEMPLATES).format(slot=mem_slot, new_val=new_val) + '\n'
        middle = _make_distractors(rng, d_before) + update_line + _make_distractors(rng, d_after)

        slots2 = {k: rng.choice(SLOT_VALUES[k]) for k in SLOT_NAMES}
        slots2[mem_slot] = new_val
        remaining = [k for k in SLOT_NAMES if k != mem_slot]
        known2 = rng.choice(remaining)
        missing2 = [k for k in remaining if k != known2][0]
        plan2 = PRIORITY_TO_PLAN[slots2['優先軸']]
        prompt = turn1 + middle + rng.choice(CONSULT_TEMPLATES).format(topic=topic2, known_text=_fmt_known(slots2, [known2])) + '\n'
        ideal = (
            f'<think>テーマ: {topic2}。既知: {known2}={slots2[known2]}。更新記憶: {mem_slot}={new_val}。不足: {missing2}</think>'
            f'<ask>{missing2}</ask><user>{missing2}: {slots2[missing2]}</user>'
            f'<think>整理完了。全前提が揃った。</think>'
            f'<summary>{_build_summary(topic2, slots2, plan2)}</summary><|eos|>'
        )
        return Episode(3, prompt, topic2, slots2, [known2], [mem_slot], {mem_slot: old_val}, [missing2], plan2, ideal, d_before + d_after)

    # Level 4: Composite 2-slot recall + overwrite + long distractors
    n_dist = num_distractors if num_distractors is not None else rng.randint(6, 12)
    d_before = max(1, n_dist // 2)
    d_after = max(1, n_dist - d_before)
    upd_slot = rng.choice(['制約', '優先軸'])
    keep_slot = '優先軸' if upd_slot == '制約' else '制約'
    old_val, new_val = rng.sample(SLOT_VALUES[upd_slot], 2)
    keep_val = rng.choice(SLOT_VALUES[keep_slot])
    slots1 = {'目的': rng.choice(SLOT_VALUES['目的']), upd_slot: old_val, keep_slot: keep_val}
    turn1 = _build_turn1(rng, topic1, slots1, asked_slot=keep_slot)
    update_line = rng.choice(UPDATE_TEMPLATES).format(slot=upd_slot, new_val=new_val) + '\n'
    middle = _make_distractors(rng, d_before) + update_line + _make_distractors(rng, d_after)

    slots2 = {'目的': rng.choice(SLOT_VALUES['目的']), upd_slot: new_val, keep_slot: keep_val}
    plan2 = PRIORITY_TO_PLAN[slots2['優先軸']]
    prompt = turn1 + middle + rng.choice(CONSULT_NO_KNOWN_TEMPLATES).format(topic=topic2) + '\n'
    ideal = (
        f'<think>テーマ: {topic2}。記憶: {keep_slot}={keep_val}。更新記憶: {upd_slot}={new_val}。不足: 目的</think>'
        f'<ask>目的</ask><user>目的: {slots2["目的"]}</user>'
        f'<think>整理完了。全前提が揃った。</think>'
        f'<summary>{_build_summary(topic2, slots2, plan2)}</summary><|eos|>'
    )
    return Episode(4, prompt, topic2, slots2, [], ['制約', '優先軸'], {upd_slot: old_val}, ['目的'], plan2, ideal, d_before + d_after)

tokenizer = ThinkingTokenizer.build_default()
(OUTPUT_DIR / 'vocab.json').write_text(json.dumps(tokenizer.token_to_id, ensure_ascii=False, indent=2), encoding='utf-8')
print('Vocab size:', tokenizer.vocab_size)
demo_rng = random.Random(0)
for lvl, d in [(1, 2), (2, 12), (3, 8), (4, 10)]:
    ep = generate_episode(lvl, demo_rng, num_distractors=d)
    n_tok = len(tokenizer.encode(ep.prompt + ep.ideal_completion))
    print(f'\n--- Level {lvl} (d={d}, total_tokens={n_tok}, mem={ep.memory_slots}, upd={ep.updated_slots}) ---')
    print('  Turn2:', ep.prompt.strip().splitlines()[-1])
    print('  Ideal:', ep.ideal_completion)

In [ ]:
# 4. Titans (MAC) モデル構築 & Stage 1 (Curriculum SFT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    major, _ = torch.cuda.get_device_capability()
    amp_dtype = torch.bfloat16 if (major >= 8 and torch.cuda.is_bf16_supported()) else torch.float16
else:
    amp_dtype = torch.float32
use_amp = device.type == 'cuda' and amp_dtype in (torch.bfloat16, torch.float16)
use_scaler = device.type == 'cuda' and amp_dtype == torch.float16

model = MemoryAsContextTransformer(
    num_tokens=tokenizer.vocab_size,
    dim=cfg['dim'],
    depth=cfg['depth'],
    segment_len=cfg['segment_len'],
    num_persist_mem_tokens=cfg['num_persist_mem_tokens'],
    num_longterm_mem_tokens=cfg['num_longterm_mem_tokens'],
).to(device)
print(f'Titans (MAC) params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | device={device} | amp={amp_dtype}')

def tokenize_sft_episode(ep, max_seq_len):
    p_ids = tokenizer.encode(ep.prompt)
    c_ids = tokenizer.encode(ep.ideal_completion)
    ids = p_ids + c_ids
    labels = [-100] * len(p_ids) + list(c_ids)
    u_start, u_end = tokenizer.token_to_id['<user>'], tokenizer.token_to_id['</user>']
    in_user = False
    for i in range(len(p_ids), len(labels)):
        if ids[i] == u_start:
            in_user = True
            labels[i] = -100
        elif in_user:
            labels[i] = -100
            if ids[i] == u_end:
                in_user = False
    return ids[:max_seq_len], labels[:max_seq_len]

def sample_sft_episode(progress, rng):
    r = rng.random()
    if progress < 0.35:
        if r < 0.55: return generate_episode(1, rng, num_distractors=rng.randint(1, 2))
        if r < 0.85: return generate_episode(2, rng, num_distractors=rng.randint(4, 7))
        return generate_episode(3, rng, num_distractors=rng.randint(2, 5))
    if progress < 0.70:
        if r < 0.20: return generate_episode(1, rng, num_distractors=rng.randint(1, 2))
        if r < 0.50: return generate_episode(2, rng, num_distractors=rng.randint(5, 8))
        if r < 0.85: return generate_episode(3, rng, num_distractors=rng.randint(4, 7))
        return generate_episode(4, rng, num_distractors=rng.randint(4, 6))
    if r < 0.15: return generate_episode(1, rng, num_distractors=rng.randint(1, 2))
    if r < 0.45: return generate_episode(2, rng, num_distractors=rng.randint(6, 8))
    if r < 0.80: return generate_episode(3, rng, num_distractors=rng.randint(4, 8))
    return generate_episode(4, rng, num_distractors=rng.randint(4, 8))

def collate(items):
    max_len = max(len(ids) for ids, _ in items)
    x = torch.full((len(items), max_len), tokenizer.pad_token_id, dtype=torch.long)
    y = torch.full((len(items), max_len), -100, dtype=torch.long)
    for i, (ids, lbls) in enumerate(items):
        x[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        y[i, :len(lbls)] = torch.tensor(lbls, dtype=torch.long)
    return x, y

def cosine_lr(step, warmup, total, max_lr, min_ratio=0.1):
    if step < warmup:
        return max_lr * (step + 1) / max(1, warmup)
    p = min(max((step - warmup) / max(1, total - warmup), 0.0), 1.0)
    return max_lr * (min_ratio + (1.0 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * p)))

rng = random.Random(SEED)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['sft_lr'], betas=(0.9, 0.95), weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler) if hasattr(torch.amp, 'GradScaler') else torch.cuda.amp.GradScaler(enabled=use_scaler)
warmup = max(1, min(50, SFT_STEPS // 10))

model.train()
t0 = time.time()
for step in range(1, SFT_STEPS + 1):
    progress = (step - 1) / max(1, SFT_STEPS)
    lr = cosine_lr(step - 1, warmup, SFT_STEPS, cfg['sft_lr'])
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    for _ in range(cfg['sft_grad_accum']):
        batch = [tokenize_sft_episode(sample_sft_episode(progress, rng), cfg['seq_len']) for _ in range(cfg['sft_batch_size'])]
        x, y = collate(batch)
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        ctx = torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()
        with ctx:
            logits = model(x[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1), ignore_index=-100)
            scaled_loss = loss / cfg['sft_grad_accum']
        if use_scaler:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
        accum_loss += float(loss.item()) / cfg['sft_grad_accum']
    if use_scaler:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    if step % 50 == 0 or step == 1 or step == SFT_STEPS:
        print(f'[SFT] step {step:4d}/{SFT_STEPS} | loss {accum_loss:.4f} | lr {lr:.2e} | {time.time()-t0:.1f}s')

torch.save({'stage': 'sft', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage1_sft_latest.pt')
if device.type == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
# 5. 対話ロールアウト・厳格スコア関数・4レベル評価 & 距離別劣化曲線
ASK_RE = re.compile(r'<ask>([^<>]+)</ask>')
SUMMARY_RE = re.compile(r'<summary>(.*?)</summary>', re.DOTALL)
THINK_RE = re.compile(r'<think>.*?</think>', re.DOTALL)

def parse_summary_strict(body):
    parts = body.split('|')
    parsed = {}
    clean = len(parts) == 5
    for seg in parts:
        seg = seg.strip()
        if not seg or '=' not in seg:
            clean = False
            continue
        k, v = seg.split('=', 1)
        k, v = k.strip(), v.strip()
        if not k or not v or k in parsed:
            clean = False
        parsed[k] = v
    return parsed, clean

def score_rollout(ep, text, asks):
    has_think = bool(THINK_RE.search(text))
    sm = SUMMARY_RE.search(text)
    parsed, clean_pipes = parse_summary_strict(sm.group(1)) if sm else ({}, False)
    format_ok = has_think and (sm is not None) and clean_pipes
    reward = 0.20 if format_ok else (0.05 if (has_think and sm is not None) else 0.0)

    missing_set = set(ep.missing_slots)
    known_or_mem = set(ep.known_slots) | set(ep.memory_slots)
    seen, hit, redundant, invalid, reasked_mem = set(), set(), 0, 0, False
    for q in asks:
        if q not in SLOT_NAMES:
            invalid += 1
            continue
        if q in seen:
            redundant += 1
            continue
        seen.add(q)
        if q in missing_set:
            hit.add(q)
        elif q in known_or_mem:
            redundant += 1
            if q in ep.memory_slots:
                reasked_mem = True

    if missing_set:
        reward += 0.40 * (len(hit) / len(missing_set))
    reward -= 0.25 * redundant + 0.20 * invalid
    if len(asks) > 2:
        reward -= 0.20
    ask_exact = (hit == missing_set) and (redundant == 0) and (invalid == 0)

    expected = {'テーマ': ep.topic, '目的': ep.full_slots['目的'], '制約': ep.full_slots['制約'], '優先軸': ep.full_slots['優先軸'], '方針': ep.expected_plan}
    matched = sum(1 for k, v in expected.items() if parsed.get(k) == v)
    slot_acc = matched / len(expected)
    reward += 1.00 * slot_acc
    correct = (matched == len(expected)) and format_ok and ask_exact

    all_mem_ok = all(parsed.get(k) == ep.full_slots[k] for k in ep.memory_slots)
    mem_reused = bool(ep.memory_slots) and all_mem_ok and (not reasked_mem)
    if mem_reused:
        reward += 0.25

    used_stale, overwrite_ok = False, False
    if ep.updated_slots:
        used_stale = any(parsed.get(k) == old_v for k, old_v in ep.updated_slots.items())
        overwrite_ok = all(parsed.get(k) == ep.full_slots[k] for k in ep.updated_slots) and (not reasked_mem)
        if overwrite_ok:
            reward += 0.30
        if used_stale:
            reward -= 0.35

    return reward, correct, slot_acc, ask_exact, mem_reused, overwrite_ok, used_stale, format_ok

@torch.no_grad()
def rollout(ep, greedy=False, temp=1.0):
    model.eval()
    p_ids = tokenizer.encode(ep.prompt)
    if len(p_ids) >= cfg['seq_len'] - 16:
        p_ids = p_ids[-(cfg['seq_len'] - 16):]
    seq = list(p_ids)
    mask = [0] * len(p_ids)
    asks = []
    for _ in range(MAX_NEW_TOKENS):
        if len(seq) >= cfg['seq_len']:
            break
        logits = model(torch.tensor([seq], dtype=torch.long, device=device))[:, -1, :]
        if greedy or temp <= 1e-5:
            nxt = int(torch.argmax(logits, dim=-1).item())
        else:
            probs = F.softmax(logits / temp, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            rm = torch.cumsum(sp, dim=-1) > 0.92
            rm[..., 1:] = rm[..., :-1].clone()
            rm[..., 0] = False
            sp[rm] = 0.0
            sp = sp / sp.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            nxt = int(si.gather(-1, torch.multinomial(sp, 1)).item())
        seq.append(nxt)
        mask.append(1)
        if nxt == tokenizer.eos_token_id:
            break
        if nxt == tokenizer.ask_end_id and len(asks) < 2:
            m = ASK_RE.findall(tokenizer.decode(seq[len(p_ids):]))
            if m:
                q = m[-1].strip()
                asks.append(q)
                reply = f'{q}: {ep.full_slots[q]}' if q in ep.full_slots else 'エラー: 該当なし'
                e_ids = tokenizer.encode(f'<user>{reply}</user>')
                if len(seq) + len(e_ids) < cfg['seq_len']:
                    seq.extend(e_ids)
                    mask.extend([0] * len(e_ids))
    text = tokenizer.decode(seq[len(p_ids):])
    r, ok, s_acc, a_ok, m_ok, ow_ok, stale, f_ok = score_rollout(ep, text, asks)
    return dict(full_ids=seq, action_mask=mask, text=text, asks=asks, reward=r, correct=ok, slot_acc=s_acc, ask_exact=a_ok, mem_reused=m_ok, overwrite_ok=ow_ok, stale=stale, format_ok=f_ok)

def evaluate_levels(n_per_level=EVAL_PER_LEVEL, seed=9999):
    erng = random.Random(seed)
    specs = {1: ('L1_short_d2', 2), 2: ('L2_long_d12', 12), 3: ('L3_overwrite_d8', 8), 4: ('L4_composite_d10', 10)}
    out = {}
    for lvl, (label, d) in specs.items():
        c = fmt = ask_ok = mem_ok = ow_ok = stale_cnt = 0
        s_sum = r_sum = tok_sum = 0.0
        shown = False
        for _ in range(n_per_level):
            ep = generate_episode(lvl, erng, num_distractors=d)
            tok_sum += len(tokenizer.encode(ep.prompt))
            ro = rollout(ep, greedy=True)
            c += int(ro['correct']); fmt += int(ro['format_ok']); ask_ok += int(ro['ask_exact'])
            mem_ok += int(ro['mem_reused']); ow_ok += int(ro['overwrite_ok']); stale_cnt += int(ro['stale'])
            s_sum += ro['slot_acc']; r_sum += ro['reward']
            if not shown:
                shown = True
                print(f'  [{label}] mem={ep.memory_slots} upd={ep.updated_slots} | asks={ro["asks"]} | ok={ro["correct"]} | stale={ro["stale"]}')
                print(f'    出力: {ro["text"]}')
        entry = dict(
            avg_prompt_tokens=round(tok_sum / n_per_level, 1),
            accuracy=round(c / n_per_level, 3),
            slot_accuracy=round(s_sum / n_per_level, 3),
            ask_exact_rate=round(ask_ok / n_per_level, 3),
            memory_reuse_rate=round(mem_ok / n_per_level, 3),
            format_rate=round(fmt / n_per_level, 3),
            avg_reward=round(r_sum / n_per_level, 3),
        )
        if lvl in (3, 4):
            entry['overwrite_rate'] = round(ow_ok / n_per_level, 3)
            entry['stale_memory_rate'] = round(stale_cnt / n_per_level, 3)
        out[label] = entry
    return out

def evaluate_distance_curve(distances=(2, 6, 12, 18), n_per_dist=EVAL_PER_DIST, seed=12345):
    erng = random.Random(seed)
    curve = {}
    for d in distances:
        c = mem_ok = 0
        tok_sum = 0.0
        for _ in range(n_per_dist):
            ep = generate_episode(2, erng, num_distractors=d)
            tok_sum += len(tokenizer.encode(ep.prompt))
            ro = rollout(ep, greedy=True)
            c += int(ro['correct']); mem_ok += int(ro['mem_reused'])
        curve[f'd={d}'] = dict(avg_prompt_tokens=round(tok_sum / n_per_dist, 1), accuracy=round(c / n_per_dist, 3), memory_reuse_rate=round(mem_ok / n_per_dist, 3))
    return curve

print('\n=== Stage 1 (SFT) 完了後の評価 ===')
metrics_sft = evaluate_levels()
print(json.dumps(metrics_sft, ensure_ascii=False, indent=2))
print('\n=== Stage 1 (SFT) 距離別記憶保持カーブ (Level 2) ===')
curve_sft = evaluate_distance_curve()
print(json.dumps(curve_sft, ensure_ascii=False, indent=2))

In [ ]:
# 6. Stage 2 (GRPO): 長距離 L2 (d=10..16) / 上書き L3 / 複合 L4 の重点強化学習
ref_model = copy.deepcopy(model).to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

def seq_logprobs(m, full_ids, action_mask):
    x = torch.tensor([full_ids[:-1]], dtype=torch.long, device=device)
    tgt = torch.tensor([full_ids[1:]], dtype=torch.long, device=device)
    msk = torch.tensor([action_mask[1:]], dtype=torch.float32, device=device)
    ctx = torch.amp.autocast(device_type=device.type, dtype=amp_dtype) if use_amp else nullcontext()
    with ctx:
        lp = F.log_softmax(m(x).float(), dim=-1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return lp.squeeze(0), msk.squeeze(0)

def sample_grpo_episode(rng):
    r = rng.random()
    if r < 0.30: return generate_episode(2, rng, num_distractors=rng.randint(10, 16))
    if r < 0.60: return generate_episode(3, rng, num_distractors=rng.randint(8, 14))
    return generate_episode(4, rng, num_distractors=rng.randint(8, 14))

grpo_opt = torch.optim.AdamW(model.parameters(), lr=cfg['grpo_lr'], betas=(0.9, 0.95), weight_decay=0.01)
grpo_rng = random.Random(SEED + 1000)
ema_r, updated_steps = 0.0, 0
t0 = time.time()

for step in range(1, GRPO_STEPS + 1):
    ep = sample_grpo_episode(grpo_rng)
    ros = [rollout(ep, greedy=False, temp=GRPO_TEMP) for _ in range(GROUP_SIZE)]
    rs = [ro['reward'] for ro in ros]
    mean_r = sum(rs) / len(rs)
    std_r = math.sqrt(sum((r - mean_r) ** 2 for r in rs) / len(rs))
    acc_g = sum(1.0 for ro in ros if ro['correct']) / len(ros)
    ema_r = 0.9 * ema_r + 0.1 * mean_r if step > 1 else mean_r

    if std_r < 1e-6:
        if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
            print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{ep.level}(d={ep.num_distractors:2d}) | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} | upd={updated_steps}/{step} | skip(zero-var)')
        continue

    updated_steps += 1
    advs = [(r - mean_r) / (std_r + 1e-6) for r in rs]
    model.train()
    grpo_opt.zero_grad(set_to_none=True)
    total_kl = 0.0
    for ro, adv in zip(ros, advs):
        if sum(ro['action_mask'][1:]) == 0:
            continue
        pol_lp, msk = seq_logprobs(model, ro['full_ids'], ro['action_mask'])
        with torch.no_grad():
            ref_lp, _ = seq_logprobs(ref_model, ro['full_ids'], ro['action_mask'])
        denom = msk.sum().clamp_min(1.0)
        lr_diff = (ref_lp - pol_lp).clamp(-10.0, 10.0)
        kl = ((torch.exp(lr_diff) - lr_diff - 1.0) * msk).sum() / denom
        pg = -adv * (pol_lp * msk).sum() / denom
        ((pg + KL_BETA * kl) / GROUP_SIZE).backward()
        total_kl += float(kl.item()) / GROUP_SIZE
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    grpo_opt.step()
    if device.type == 'cuda' and step % 10 == 0:
        torch.cuda.empty_cache()
    if step % 10 == 0 or step == 1 or step == GRPO_STEPS:
        print(f'[GRPO] step {step:3d}/{GRPO_STEPS} | L{ep.level}(d={ep.num_distractors:2d}) | r={mean_r:.3f} (ema {ema_r:.3f}) | acc={acc_g:.2f} | upd={updated_steps}/{step} | kl={total_kl:.4f} | {time.time()-t0:.1f}s')

torch.save({'stage': 'grpo', 'model_state_dict': model.state_dict()}, OUTPUT_DIR / 'stage2_grpo_latest.pt')
del ref_model
if device.type == 'cuda':
    torch.cuda.empty_cache()

print(f'\n=== Stage 2 (GRPO) 完了後の評価 (更新ステップ: {updated_steps}/{GRPO_STEPS}) ===')
metrics_grpo = evaluate_levels()
print(json.dumps(metrics_grpo, ensure_ascii=False, indent=2))
print('\n=== Stage 2 (GRPO) 距離別記憶保持カーブ (Level 2) ===')
curve_grpo = evaluate_distance_curve()
print(json.dumps(curve_grpo, ensure_ascii=False, indent=2))

(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(
        dict(preset=PRESET, sft=metrics_sft, curve_sft=curve_sft, grpo=metrics_grpo, curve_grpo=curve_grpo, grpo_updated_steps=updated_steps, grpo_total_steps=GRPO_STEPS),
        ensure_ascii=False, indent=2,
    ),
    encoding='utf-8',
)

In [ ]:
# 7. 複合ストレス（Level 4: 2スロット記憶＋上書き＋長距離）の対話デモ
test_ep = generate_episode(4, random.Random(2026), num_distractors=12)
res = rollout(test_ep, greedy=True)
print('=== プロンプト長:', len(tokenizer.encode(test_ep.prompt)), 'tokens ===')
print('記憶すべきスロット:', test_ep.memory_slots, '| 上書き前の古い値:', test_ep.updated_slots)
print('正解の最新スロット:', test_ep.full_slots)
print('逆質問履歴:', res['asks'])
print(f"完全一致: {res['correct']} | 上書き成功: {res['overwrite_ok']} | 古い値誤用: {res['stale']} | 報酬: {res['reward']:.2f}")
print('思考＋整理結果:\n', res['text'])